In [1]:
!pip install -q mambapy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/40.1 kB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 1.2 MB/s eta 0:00:00


In [2]:
import os
import math
import json
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from datasets import load_from_disk
from tokenizers import Tokenizer

from mambapy.mamba import Mamba, MambaConfig

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
        ),
        "GB"
    )

PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [3]:
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", DEVICE)

SEED = 61

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

Using: cuda


In [4]:
BASE_PATH = "/kaggle/input/datasets/belovedorange/indian-legal-slm-tokenized-2048"

DATASET_PATH = os.path.join(BASE_PATH, "indian_legal_2048")
TOKENIZER_PATH = os.path.join(
    BASE_PATH,
    "indian_legal_tokenizer",
    "tokenizer.json"
)

print(DATASET_PATH)
print(TOKENIZER_PATH)

/kaggle/input/datasets/belovedorange/indian-legal-slm-tokenized-2048/indian_legal_2048
/kaggle/input/datasets/belovedorange/indian-legal-slm-tokenized-2048/indian_legal_tokenizer/tokenizer.json


In [5]:
dataset = load_from_disk(DATASET_PATH)
print(dataset)

tokenizer = Tokenizer.from_file(TOKENIZER_PATH)
VOCAB_SIZE = tokenizer.get_vocab_size()
print("Vocabulary size:", VOCAB_SIZE)

PAD_ID = tokenizer.token_to_id("<pad>")
BOS_ID = tokenizer.token_to_id("<bos>")
EOS_ID = tokenizer.token_to_id("<eos>")

print("PAD:", PAD_ID)
print("BOS:", BOS_ID)
print("EOS:", EOS_ID)

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 155060
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 17474
    })
})
Vocabulary size: 16000
PAD: 0
BOS: 2
EOS: 3


In [6]:
CONTEXT_LENGTH = 2048

D_MODEL    = 512
NUM_LAYERS = 6
D_STATE    = 16
D_CONV     = 4

print("Context length:", CONTEXT_LENGTH)
print("d_model:", D_MODEL)
print("Layers:", NUM_LAYERS)
print("d_state:", D_STATE)
print("d_conv:", D_CONV)

Context length: 2048
d_model: 512
Layers: 6
d_state: 16
d_conv: 4


In [7]:
class LegalDataset(Dataset):

    def __init__(self, hf_dataset):
        self.dataset = hf_dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        return self.dataset[idx]["input_ids"]


def collate_fn(batch):

    max_length = max(len(s) for s in batch)

    input_ids = torch.full(
        (len(batch), max_length),
        PAD_ID,
        dtype=torch.long
    )

    for i, sequence in enumerate(batch):
        input_ids[i, :len(sequence)] = torch.tensor(
            sequence, dtype=torch.long
        )

    return input_ids


BATCH_SIZE = 1

train_dataset = LegalDataset(dataset["train"])
val_dataset   = LegalDataset(dataset["validation"])

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=2,
    pin_memory=True
)

print("Train sequences:", len(train_dataset))
print("Validation sequences:", len(val_dataset))

Train sequences: 155060
Validation sequences: 17474


In [8]:
class MambaLanguageModel(nn.Module):

    def __init__(
        self,
        vocab_size,
        d_model,
        num_layers,
        d_state,
        d_conv,
    ):
        super().__init__()

        self.d_model = d_model

        self.embedding = nn.Embedding(vocab_size, d_model)

        # expand is NOT a MambaConfig parameter in mambapy
        # d_model, n_layers, d_state, d_conv are the only ones
        mamba_config = MambaConfig(
            d_model=d_model,
            n_layers=num_layers,
            d_state=d_state,
            d_conv=d_conv,
        )

        self.mamba = Mamba(mamba_config)

        self.norm = nn.LayerNorm(d_model)

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

        self.lm_head.weight = self.embedding.weight

    def forward(self, input_ids):
        x = self.embedding(input_ids)
        x = self.mamba(x)
        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

In [9]:
model = MambaLanguageModel(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_layers=NUM_LAYERS,
    d_state=D_STATE,
    d_conv=D_CONV,
)

model = model.to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

Total parameters:     18,364,416
Trainable parameters: 18,364,416


In [10]:
batch = next(iter(train_loader)).to(DEVICE)

print("Input:", batch.shape)

with torch.no_grad():
    logits = model(batch)

print("Output:", logits.shape)
# Expected: [1, seq_len, 16000]

Input: torch.Size([1, 2048])


Output: torch.Size([1, 2048, 16000])


In [11]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

LEARNING_RATE = 1e-4
WEIGHT_DECAY  = 0.01

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

EPOCHS = 1
GRAD_ACCUMULATION_STEPS = 8

TOTAL_STEPS  = math.ceil(len(train_loader) / GRAD_ACCUMULATION_STEPS)
WARMUP_STEPS = int(0.1 * TOTAL_STEPS)

print("Total optimizer steps:", TOTAL_STEPS)
print("Warmup steps:", WARMUP_STEPS)


def lr_lambda(current_step):
    if current_step < WARMUP_STEPS:
        return current_step / max(1, WARMUP_STEPS)
    return max(
        0.0,
        (TOTAL_STEPS - current_step) /
        max(1, TOTAL_STEPS - WARMUP_STEPS)
    )


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

Total optimizer steps: 19383
Warmup steps: 1938


In [12]:
def train_one_epoch():
    model.train()
    total_loss = 0.0

    os.makedirs(
        "/kaggle/working/indian_legal_mamba",
        exist_ok=True
    )

    scaler = torch.amp.GradScaler(
        "cuda", enabled=(DEVICE.type == "cuda")
    )

    optimizer.zero_grad(set_to_none=True)

    best_loss = float('inf')

    for step, batch in enumerate(train_loader):

        batch  = batch.to(DEVICE, non_blocking=True)
        inputs = batch[:, :-1]
        labels = batch[:, 1:]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            # Mamba does NOT need a padding mask or causal mask
            # it is causal by design
            logits = model(inputs)

            loss = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                labels.reshape(-1)
            )
            loss = loss / GRAD_ACCUMULATION_STEPS

        scaler.scale(loss).backward()

        if (
            (step + 1) % GRAD_ACCUMULATION_STEPS == 0
            or (step + 1) == len(train_loader)
        ):
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(
                model.parameters(), 1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

            current_loss = loss.item() * GRAD_ACCUMULATION_STEPS
            if current_loss < best_loss:
                best_loss = current_loss
                torch.save(
                    model.state_dict(),
                    "/kaggle/working/indian_legal_mamba/best_checkpoint.pt"
                )

        total_loss += loss.item()

        if (step + 1) % 100 == 0:
            print(
                f"Step {step + 1:,} | "
                f"Loss: {loss.item() * GRAD_ACCUMULATION_STEPS:.4f} | "
                f"LR: {scheduler.get_last_lr()[0]:.2e} | "
                f"Grad norm: {grad_norm:.4f}"
            )

    return (total_loss / len(train_loader)) * GRAD_ACCUMULATION_STEPS

In [13]:
@torch.no_grad()
def evaluate():
    model.eval()
    total_loss = 0.0

    for batch in val_loader:
        batch  = batch.to(DEVICE, non_blocking=True)
        inputs = batch[:, :-1]
        labels = batch[:, 1:]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(inputs)
            loss   = criterion(
                logits.reshape(-1, VOCAB_SIZE),
                labels.reshape(-1)
            )

        total_loss += loss.item()

    return total_loss / len(val_loader)

In [14]:
for epoch in range(EPOCHS):

    print(f"\n{'='*50}")
    print(f"Epoch {epoch + 1}/{EPOCHS}")
    print(f"{'='*50}")

    train_loss = train_one_epoch()
    val_loss   = evaluate()

    train_ppl = math.exp(min(train_loss, 20))
    val_ppl   = math.exp(min(val_loss, 20))

    print(f"\nTrain loss:        {train_loss:.4f}")
    print(f"Validation loss:   {val_loss:.4f}")
    print(f"Train perplexity:  {train_ppl:.2f}")
    print(f"Val perplexity:    {val_ppl:.2f}")


Epoch 1/1


Step 100 | Loss: 508.2009 | LR: 6.19e-07 | Grad norm: 23.4702


Step 200 | Loss: 505.0443 | LR: 1.29e-06 | Grad norm: 23.4682


Step 300 | Loss: 507.8357 | LR: 1.91e-06 | Grad norm: 23.4738


Step 400 | Loss: 506.6952 | LR: 2.58e-06 | Grad norm: 23.5234


...

Step 114,300 | Loss: 4.1487 | LR: 2.92e-05 | Grad norm: 3.5410


Step 114,400 | Loss: 4.3909 | LR: 2.91e-05 | Grad norm: 4.0495


Step 114,500 | Loss: 4.9482 | LR: 2.91e-05 | Grad norm: 4.0578


Step 114,600 | Loss: 4.3924 | LR: 2.90e-05 | Grad norm: 4.9703


Step 114,700 | Loss: 4.5590 | LR: 2.89e-05 | Grad norm: 4.3890


In [ ]:
MODEL_DIR = "/kaggle/working/indian_legal_mamba"
os.makedirs(MODEL_DIR, exist_ok=True)

torch.save(
    model.state_dict(),
    os.path.join(MODEL_DIR, "mamba_model.pt")
)

config = {
    "architecture": "Mamba SSM Language Model",
    "vocab_size": VOCAB_SIZE,
    "context_length": CONTEXT_LENGTH,
    "d_model": D_MODEL,
    "num_layers": NUM_LAYERS,
    "d_state": D_STATE,
    "d_conv": D_CONV,
    "expand": EXPAND,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "seed": SEED
}

with open(os.path.join(MODEL_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=4)

print("Model saved to:", MODEL_DIR)

In [ ]:
@torch.no_grad()
def generate(prompt, max_new_tokens=100, temperature=0.8):

    model.eval()

    prompt_ids = tokenizer.encode(prompt).ids

    input_ids = torch.tensor(
        [prompt_ids], dtype=torch.long, device=DEVICE
    )

    for _ in range(max_new_tokens):

        input_ids = input_ids[:, -CONTEXT_LENGTH:]

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            logits = model(input_ids)

        next_token_logits = logits[:, -1, :] / temperature

        probabilities = F.softmax(next_token_logits, dim=-1)

        next_token = torch.multinomial(probabilities, num_samples=1)

        input_ids = torch.cat([input_ids, next_token], dim=1)

        if next_token.item() == EOS_ID:
            break

    return tokenizer.decode(
        input_ids[0].tolist()
    ).replace("Ġ", " ").replace("Ċ", "\n").strip()


prompt = "The Supreme Court of India"

generated = generate(prompt, max_new_tokens=100, temperature=0.8)
print(generated)